# Лабораторная работа № 1
## Индексирование и булев информационный поиск

**Дисциплина:** «Информационный поиск и базы данных»

**Направление:** 44.03.05 Педагогическое образование, профиль «Информатика и английский язык»

**Модуль:** 1. Семантическая информация и принципы ИПС в образовании

**Среда:** Google Colab · Python 3 · только стандартная библиотека
**Максимум:** 20 баллов (+1 за дополнительную часть)

---

### Цель работы

Освоить подготовку текстов, построение инвертированного индекса и выполнение булевых запросов
`AND` и `OR` по текстовому корпусу.

### Почему Colab

Исходная версия работы требовала локального окружения: клонировать репозиторий, поднять
виртуальное окружение, запустить скрейпер семинара 02 и получить `result.jsonl`. Виртуальная
машина есть не у всех, поэтому этот ноутбук самодостаточен: корпус собирается или порождается
прямо в Colab, внешние библиотеки не нужны, а итоговые файлы выгружаются одной ячейкой.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями.
2. Файлы `index.json` и `lab_01_results.json`, полученные в части 5.
3. `README.md` вашего репозитория с ФИО, группой, номером варианта, описанием метода,
   результатами и выводом.

### Критерии оценки — 20 баллов

| Критерий | Баллы |
|---|---|
| Корректная подготовка корпуса и инвертированный индекс | 6 |
| Корректный булев поиск `AND` / `OR` | 6 |
| Выполнение индивидуального варианта и измерения | 4 |
| GitHub-оформление, воспроизводимость и вывод | 4 |
| **Дополнительно:** аналог этапа поиска средствами PostgreSQL | +1 |

Итог за работу не превышает 20 баллов.

### Материалы

* Семинар «Индексирование» — <https://github.com/BosenkoTM/IIR/tree/main/seminars/03-indexing>
* Исходное задание HW1 — <https://github.com/BosenkoTM/IIR/tree/main/homeworks/hw1-boolean-retrieval>
* Manning C. D., Raghavan P., Schütze H. *Introduction to Information Retrieval*, гл. 1

> ### ВАРИАНТ ПРЕПОДАВАТЕЛЯ
> Полное решение всех заданий, ноутбук выполнен целиком. Эталонный прогон сделан по варианту 1.
> Студенческий вариант отличается только содержимым кодовых ячеек.
> Числовые значения зависят от источника корпуса и runtime — оценивается корректность метода и содержательность выводов.


---
## Задание 0. Индивидуальный вариант

**Укажите свой номер варианта в переменной `VARIANT_NO` ниже.** Все параметры работы —
`random_state`, `N`, `min_len` и основной оператор — берутся из таблицы автоматически.

| Вариант | random_state | N | min_len | Оператор |
|---|---|---|---|---|
| 1 | 101 | 500 | 2 | AND |
| 2 | 102 | 750 | 3 | OR |
| 3 | 103 | 1000 | 4 | AND |
| 4 | 104 | 1250 | 3 | AND |
| 5 | 105 | 1500 | 2 | OR |
| 6 | 106 | 500 | 2 | AND |
| 7 | 107 | 750 | 3 | OR |
| 8 | 108 | 1000 | 4 | AND |
| 9 | 109 | 1250 | 3 | AND |
| 10 | 110 | 1500 | 2 | OR |
| 11 | 111 | 500 | 2 | AND |
| 12 | 112 | 750 | 3 | OR |
| 13 | 113 | 1000 | 4 | AND |
| 14 | 114 | 1250 | 3 | AND |
| 15 | 115 | 1500 | 2 | OR |
| 16 | 116 | 500 | 2 | AND |
| 17 | 117 | 750 | 3 | OR |
| 18 | 118 | 1000 | 4 | AND |
| 19 | 119 | 1250 | 3 | AND |
| 20 | 120 | 1500 | 2 | OR |
| 21 | 121 | 500 | 2 | AND |
| 22 | 122 | 750 | 3 | OR |
| 23 | 123 | 1000 | 4 | AND |
| 24 | 124 | 1250 | 3 | AND |
| 25 | 125 | 1500 | 2 | OR |

In [4]:
# ========================================
STUDENT_NAME  = "Чаговец Валерия Ярославовна"
STUDENT_GROUP = "ИНФА-231"
VARIANT_NO    = 13
# ========================================

VARIANTS = {
    n: {"random_state": 100 + n,
        "N": [500, 750, 1000, 1250, 1500][(n - 1) % 5],
        "min_len": [2, 3, 4, 3, 2][(n - 1) % 5],
        "operator": ["AND", "OR", "AND", "AND", "OR"][(n - 1) % 5]}
    for n in range(1, 26)
}

assert 1 <= VARIANT_NO <= 25, "Номер варианта — целое число от 1 до 25"
V = VARIANTS[VARIANT_NO]

RANDOM_STATE = V["random_state"]
N            = V["N"]
MIN_LEN      = V["min_len"]
OPERATOR     = V["operator"]
QUERY        = "love story"   # при желании замените на свой запрос из двух-трёх слов

print(f"Студент      : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант      : {VARIANT_NO}")
print(f"random_state : {RANDOM_STATE}")
print(f"N            : {N}")
print(f"min_len      : {MIN_LEN}")
print(f"Оператор     : {OPERATOR}")
print(f"Запрос       : {QUERY!r}")

Студент      : Чаговец Валерия Ярославовна, группа ИНФА-231
Вариант      : 13
random_state : 113
N            : 1000
min_len      : 4
Оператор     : AND
Запрос       : 'love story'


In [5]:
import json
import random
import re
import time
import statistics
from collections import defaultdict, Counter
from pathlib import Path

WORK = Path("/content") if Path("/content").exists() else Path(".")
print("Рабочий каталог:", WORK)

Рабочий каталог: /content


---
# Часть 1. Корпус и инвертированный индекс (6 баллов)

## 1.1. Откуда берутся документы

Ноутбук поддерживает три источника корпуса и пробует их по очереди.

| Источник | Когда используется |
|---|---|
| `result.jsonl` | если вы уже собрали корпус на семинаре 02 и загрузили файл в Colab |
| `books.toscrape.com` | сбор прямо из ноутбука; сайт создан специально для учебного скрейпинга |
| синтетический генератор | резерв: работает без сети и даёт любое число документов |

Если вам нужен именно ваш корпус с семинара, загрузите `result.jsonl` в Colab
(панель слева → значок папки → «Загрузить») либо раскомментируйте ячейку с `files.upload()`.

> **Важно про размер.** На `books.toscrape.com` ровно 1000 книг. Вариантам с `N = 1250`
> и `N = 1500` этого не хватит, поэтому корпус при необходимости дополняется синтетическими
> документами — об этом будет напечатано предупреждение.

In [ ]:
# Раскомментируйте, если хотите загрузить собственный result.jsonl с семинара 02:
# from google.colab import files
# files.upload()

## Упражнение 1.2. Чтение JSONL

Реализуйте функцию `load_documents(path)`. Файл `result.jsonl` содержит по одному
JSON-объекту в строке. Скрейпер семинара 02 оборачивает данные в конверт `FileSink`:

```json
{"tries": 0, "result": {"title": "...", "description": "...", "genre": "..."}, "error": null}
```

Функция должна:

1. пропускать пустые строки;
2. распаковывать конверт, если есть ключ `result`; строки с непустым `error` пропускать;
3. возвращать список словарей с ключами `source_id`, `title`, `description`, `genre`, `price`
   и `text`, где `text = title + " " + description + " " + genre`.

In [6]:
def load_documents(path) -> list[dict]:
    """Читает JSONL-корпус и возвращает список документов."""
    documents = []
    with open(path, "r", encoding="utf-8") as f:
        for source_id, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            if "result" in obj:                       # конверт FileSink
                if obj.get("error") is not None or not obj.get("result"):
                    continue
                item = obj["result"]
            else:                                     # «плоский» JSONL
                item = obj

            title = str(item.get("title", ""))
            description = str(item.get("description", ""))
            genre = str(item.get("genre", ""))

            documents.append({
                "source_id": source_id,
                "title": title,
                "description": description,
                "genre": genre,
                "price": item.get("price"),
                "text": f"{title} {description} {genre}",
            })
    return documents


# самопроверка на маленьком файле
demo = WORK / "_demo.jsonl"
demo.write_text(
    '{"tries":0,"result":{"title":"Magic Book","description":"A world of magic",'
    '"genre":"Fantasy","price":10.0},"error":null}\n'
    '\n'
    '{"tries":1,"result":null,"error":"timeout"}\n',
    encoding="utf-8",
)
_d = load_documents(demo)
print(_d)
assert len(_d) == 1, "Строка с ошибкой и пустая строка должны быть пропущены"
assert _d[0]["text"] == "Magic Book A world of magic Fantasy"

[{'source_id': 0, 'title': 'Magic Book', 'description': 'A world of magic', 'genre': 'Fantasy', 'price': 10.0, 'text': 'Magic Book A world of magic Fantasy'}]


## 1.3. Сбор корпуса

Ячейка ниже готова — читать её полезно, но переписывать не нужно. Она по очереди пробует
три источника и при необходимости дополняет корпус до нужного размера.

In [7]:
GENRES = ["Fantasy", "Science Fiction", "Mystery", "Historical Fiction", "Poetry",
          "Travel", "Philosophy", "Biography", "Childrens", "Romance",
          "Thriller", "Horror", "Classics", "Art", "Music", "Science", "History"]

# Ядро словаря: частотные служебные и содержательные слова, в том числе короткие —
# именно они отсекаются фильтром min_len.
CORE = """the of and to in is it on at by or as be he she we my an for with from
that this not are was were has have had who what when where why how all any
book story world magic life time day night man woman child house door road
ancient kingdom shadow light forest river mountain ocean secret journey voyage
island castle dragon knight witch wizard spell memory silence winter summer
autumn garden letter mirror clock window science machine planet station engineer
signal orbit gravity comet robot murder detective evidence witness alibi verdict
prison escape chase clue love promise wedding stranger neighbour village harbour
lighthouse cottage history empire revolution treaty battle general soldier
archive manuscript museum poem verse rhythm metaphor language grammar dictionary
translation accent teacher student lesson school library reading writing question
answer knowledge map compass railway desert canyon glacier volcano jungle savanna""".split()


def _make_vocabulary(n_tail: int = 6000, seed: int = 7) -> list[str]:
    """CORE + процедурно порождённый «хвост» редких слов разной длины."""
    rng = random.Random(seed)
    cons = "bcdfghjklmnprstvwz"
    vows = "aeiouy"
    tail, seen = [], set(CORE)
    while len(tail) < n_tail:
        syllables = rng.randint(1, 4)
        w = "".join(rng.choice(cons) + rng.choice(vows) +
                    (rng.choice(cons) if rng.random() < 0.45 else "")
                    for _ in range(syllables))
        if 2 <= len(w) <= 12 and w not in seen:
            seen.add(w)
            tail.append(w)
    return CORE + tail


VOCAB = _make_vocabulary()
# Веса по закону Ципфа: слово ранга r встречается примерно в 1/r раз реже первого.
ZIPF_WEIGHTS = [1.0 / (r + 2.5) ** 1.07 for r in range(1, len(VOCAB) + 1)]


def make_synthetic_corpus(n_docs: int, seed: int = 2026) -> list[dict]:
    """Детерминированный учебный корпус с ципфовским распределением слов.

    Работает без сети, даёт любое число документов и воспроизводит два свойства
    настоящего текста: сублинейный рост словаря (закон Хипса) и большую долю слов,
    встретившихся ровно один раз.
    """
    rng = random.Random(seed)
    head = VOCAB[:400]                       # частотное ядро для заголовков
    docs = []
    for i in range(n_docs):
        title = " ".join(w.capitalize() for w in rng.sample(head, rng.randint(2, 4)))
        body = rng.choices(VOCAB, weights=ZIPF_WEIGHTS, k=rng.randint(60, 140))
        description = " ".join(body).capitalize() + "."
        genre = rng.choice(GENRES)
        docs.append({
            "source_id": i,
            "title": title,
            "description": description,
            "genre": genre,
            "price": round(rng.uniform(10, 60), 2),
            "text": f"{title} {description} {genre}",
        })
    return docs


def scrape_books_toscrape(limit: int = 1000, timeout: int = 20) -> list[dict]:
    """Сбор корпуса с учебного сайта books.toscrape.com (только стандартная библиотека)."""
    from urllib.request import urlopen
    from urllib.parse import urljoin
    from html.parser import HTMLParser

    BASE = "https://books.toscrape.com/catalogue/"
    docs, page = [], 1

    def fetch(url):
        with urlopen(url, timeout=timeout) as r:
            return r.read().decode("utf-8", errors="ignore")

    while len(docs) < limit and page <= 50:
        html = fetch(f"{BASE}page-{page}.html")
        hrefs = re.findall(r'<h3><a href="([^"]+)"', html)
        for href in hrefs:
            if len(docs) >= limit:
                break
            book = fetch(urljoin(BASE, href))
            title = re.search(r"<h1>(.*?)</h1>", book)
            desc = re.search(r'<div id="product_description".*?<p>(.*?)</p>', book, re.S)
            genre = re.findall(r'<li><a href="\.\./category/books/[^"]+">(.*?)</a></li>', book)
            price = re.search(r"£([\d.]+)", book)
            docs.append({
                "source_id": len(docs),
                "title": (title.group(1) if title else "").strip(),
                "description": re.sub(r"<[^>]+>", " ", desc.group(1)) if desc else "",
                "genre": genre[0].strip() if genre else "",
                "price": float(price.group(1)) if price else None,
                "text": "",
            })
            docs[-1]["text"] = (f'{docs[-1]["title"]} {docs[-1]["description"]} '
                                f'{docs[-1]["genre"]}')
        page += 1
    return docs


def get_corpus(min_docs: int) -> tuple[list[dict], str]:
    """Возвращает (корпус, название источника), гарантируя размер не меньше min_docs."""
    jsonl = WORK / "result.jsonl"
    if jsonl.exists():
        docs = load_documents(jsonl)
        source = "result.jsonl (семинар 02)"
        print(f"Источник: {source}, документов {len(docs)}")
    else:
        try:
            print("Файл result.jsonl не найден, пробую собрать books.toscrape.com ...")
            docs = scrape_books_toscrape(limit=1000)
            source = "books.toscrape.com"
            print(f"Источник: {source}, документов {len(docs)}")
        except Exception as exc:
            print(f"Сбор не удался ({type(exc).__name__}), перехожу к синтетическому корпусу.")
            docs, source = [], "синтетический корпус"

    if len(docs) < min_docs:
        need = min_docs - len(docs) + 200          # запас для экспериментов
        if docs:
            print(f"ВНИМАНИЕ: документов {len(docs)}, требуется {min_docs}. "
                  f"Дополняю {need} синтетическими.")
            source += " + синтетический корпус"
        extra = make_synthetic_corpus(need)
        for d in extra:
            d["source_id"] += len(docs)
        docs = docs + extra

    return docs, source


t0 = time.perf_counter()
docs_all, CORPUS_SOURCE = get_corpus(min_docs=N + 200)
corpus_time = time.perf_counter() - t0

print(f"\nВсего документов : {len(docs_all)}")
print(f"Источник         : {CORPUS_SOURCE}")
print(f"Время подготовки : {corpus_time:.2f} с")
print("\nПример документа:")
print(json.dumps(docs_all[0], ensure_ascii=False, indent=2)[:400])

Файл result.jsonl не найден, пробую собрать books.toscrape.com ...
Источник: books.toscrape.com, документов 1000
ВНИМАНИЕ: документов 1000, требуется 1200. Дополняю 400 синтетическими.

Всего документов : 1400
Источник         : books.toscrape.com + синтетический корпус
Время подготовки : 286.75 с

Пример документа:
{
  "source_id": 0,
  "title": "A Light in the Attic",
  "description": "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read thes


## Упражнение 1.4. Токенизация

Реализуйте `tokenize(text, min_len)`:

1. привести текст к нижнему регистру;
2. выделить слова регулярным выражением `TOKEN_RE` (латиница, допускается апостроф внутри слова);
3. отбросить токены короче `min_len`;
4. пустой или `None` на входе — вернуть пустой список.

Это тот самый «лингвистический модуль» из лекции: приведение регистра и фильтрация —
простейшая нормализация.

In [8]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize(text: str, min_len: int = 2) -> list[str]:
    """Нижний регистр -> выделение слов -> фильтрация по длине."""
    return [
        token
        for token in TOKEN_RE.findall((text or "").lower())
        if len(token) >= min_len
    ]


print(tokenize("Magic World, the Ancient Kingdom!", min_len=2))
print(tokenize("Magic World, the Ancient Kingdom!", min_len=4))

assert tokenize("A Magic World", 2) == ["magic", "world"]
assert tokenize("A Magic World", 6) == []
assert tokenize(None, 2) == []
assert tokenize("Don't stop", 2) == ["don't", "stop"]

['magic', 'world', 'the', 'ancient', 'kingdom']
['magic', 'world', 'ancient', 'kingdom']


## Упражнение 1.5. Воспроизводимая выборка

Реализуйте `sample_documents(documents, n, random_state)`:

* `n <= 0` — вернуть весь корпус;
* если документов меньше `n` — понятное исключение `ValueError`;
* иначе — выборка без повторов через `random.Random(random_state).sample`.

Использование локального объекта `Random`, а не глобального `random.seed`, — принципиально:
результат не зависит от того, что делали в других ячейках, и работа воспроизводима.

In [9]:
def sample_documents(documents: list[dict], n: int, random_state: int) -> list[dict]:
    """Воспроизводимая случайная выборка из корпуса."""
    if n <= 0:
        return list(documents)
    if len(documents) < n:
        raise ValueError(
            f"В корпусе только {len(documents)} документов, а требуется N={n}."
        )
    rng = random.Random(random_state)
    return rng.sample(documents, n)


docs = sample_documents(docs_all, n=N, random_state=RANDOM_STATE)
print(f"В выборке {len(docs)} документов")

# воспроизводимость: тот же random_state -> та же выборка
again = sample_documents(docs_all, n=N, random_state=RANDOM_STATE)
assert [d["source_id"] for d in docs] == [d["source_id"] for d in again]
other = sample_documents(docs_all, n=N, random_state=RANDOM_STATE + 1)
assert [d["source_id"] for d in docs] != [d["source_id"] for d in other]
assert len(docs) == N

В выборке 1000 документов


## Упражнение 1.6. Инвертированный индекс

Реализуйте `build_inverted_index(documents, min_len)` — словарь
`термин -> множество doc_id`, где `doc_id` — позиция документа в выборке.

**Важная деталь.** Документ должен попасть в posting list термина ровно один раз, даже если
слово встретилось в тексте двадцать раз. Подсказка: примените `set()` к списку токенов
документа до добавления в индекс.

In [10]:
def build_inverted_index(documents: list[dict], min_len: int) -> dict[str, set[int]]:
    """Строит индекс: термин -> множество идентификаторов документов."""
    index: dict[str, set[int]] = defaultdict(set)
    for doc_id, document in enumerate(documents):
        # set() — чтобы документ попал в posting list термина ровно один раз
        for term in set(tokenize(document["text"], min_len=min_len)):
            index[term].add(doc_id)
    return dict(index)


t0 = time.perf_counter()
index = build_inverted_index(docs, MIN_LEN)
BUILD_TIME = time.perf_counter() - t0

print(f"Размер словаря           : {len(index)}")
print(f"Время построения индекса : {BUILD_TIME:.6f} с")
print(f"Всего вхождений          : {sum(len(v) for v in index.values())}")

sample_term = sorted(index)[len(index) // 2]
print(f"\nПример: {sample_term!r} -> {sorted(index[sample_term])[:10]} ...")

assert isinstance(index, dict) and index
assert all(isinstance(v, set) for v in index.values())
assert all(len(t) >= MIN_LEN for t in index), "В индексе не должно быть коротких токенов"
assert max(max(v) for v in index.values()) < len(docs)

Размер словаря           : 18764
Время построения индекса : 0.183913 с
Всего вхождений          : 83346

Пример: 'lecturer' -> [121] ...


## 1.7. Что внутри словаря

Ячейка ниже готова: она показывает десять самых частых и десять самых редких терминов.
Обратите внимание на распределение — оно иллюстрирует закон Ципфа из лекции.

In [11]:
df_pairs = sorted(((len(ids), term) for term, ids in index.items()), reverse=True)

print("Самые частые термины (документная частота):")
for freq, term in df_pairs[:10]:
    print(f"  {term:<18} df = {freq:>5}  ({freq / len(docs):.1%} документов)")

hapax = sum(1 for freq, _ in df_pairs if freq == 1)
print(f"\nТерминов, встретившихся ровно в одном документе: {hapax} "
      f"({hapax / len(index):.1%} словаря)")

Самые частые термины (документная частота):
  more               df =   682  (68.2% документов)
  that               df =   663  (66.3% документов)
  with               df =   642  (64.2% документов)
  from               df =   554  (55.4% документов)
  this               df =   435  (43.5% документов)
  when               df =   384  (38.4% документов)
  life               df =   360  (36.0% документов)
  have               df =   348  (34.8% документов)
  what               df =   338  (33.8% документов)
  world              df =   306  (30.6% документов)

Терминов, встретившихся ровно в одном документе: 10004 (53.3% словаря)


---
# Часть 2. Булев поиск (6 баллов)

## Упражнение 2.1. AND и OR

Реализуйте `boolean_search(query, index, operator, min_len)`. Функция возвращает кортеж
`(список терминов запроса, отсортированный список doc_id)`.

* токенизировать запрос теми же правилами, что и документы;
* пустой запрос — вернуть `([], [])`;
* термин, которого нет в индексе, даёт пустой posting list;
* `AND` — пересечение списков, `OR` — объединение;
* иной оператор — `ValueError`.

**Подумайте, прежде чем писать:** что должен вернуть `AND`-запрос, если одного из терминов
нет в словаре? А `OR`-запрос?

In [12]:
def boolean_search(query: str, index: dict[str, set[int]],
                   operator: str, min_len: int) -> tuple[list[str], list[int]]:
    """Булев поиск по инвертированному индексу."""
    terms = tokenize(query, min_len=min_len)
    if not terms:
        return [], []

    postings = [index.get(term, set()) for term in terms]
    operator = operator.upper()

    if operator == "AND":
        result = set.intersection(*postings)
    elif operator == "OR":
        result = set.union(*postings)
    else:
        raise ValueError("operator должен быть AND или OR")

    return terms, sorted(result)


# --- проверка на игрушечном индексе из лекции
toy = {"magic": {2, 8, 15}, "world": {2, 5, 8, 21}}
assert boolean_search("magic world", toy, "AND", 2) == (["magic", "world"], [2, 8])
assert boolean_search("magic world", toy, "OR", 2)  == (["magic", "world"], [2, 5, 8, 15, 21])
assert boolean_search("magic dragon", toy, "AND", 2)[1] == []      # dragon нет в индексе
assert boolean_search("magic dragon", toy, "OR", 2)[1] == [2, 8, 15]
assert boolean_search("", toy, "AND", 2) == ([], [])

try:
    boolean_search("magic", toy, "XOR", 2)
    raise AssertionError("Должно быть выброшено ValueError")
except ValueError:
    pass

print("Все проверки пройдены")

Все проверки пройдены


## Упражнение 2.2. Индекс против полного перебора

Напишите функцию `linear_search(query, documents, operator, min_len)`, которая решает ту же
задачу **без индекса** — последовательно просматривая тексты всех документов, как `grep`
из лекции.

Затем сравните время: выполните каждый вариант поиска не менее 20 раз и возьмите медиану
(одиночный запуск слишком быстр, чтобы его можно было измерить надёжно).

In [13]:
def linear_search(query: str, documents: list[dict],
                  operator: str, min_len: int) -> list[int]:
    """Тот же поиск, но полным просмотром корпуса — без индекса."""
    terms = tokenize(query, min_len=min_len)
    if not terms:
        return []

    operator = operator.upper()
    found = []
    for doc_id, document in enumerate(documents):
        doc_terms = set(tokenize(document["text"], min_len=min_len))
        if operator == "AND":
            hit = all(term in doc_terms for term in terms)
        elif operator == "OR":
            hit = any(term in doc_terms for term in terms)
        else:
            raise ValueError("operator должен быть AND или OR")
        if hit:
            found.append(doc_id)
    return found


def median_time(fn, repeats: int = 20) -> float:
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


t_index  = median_time(lambda: boolean_search(QUERY, index, OPERATOR, MIN_LEN))
t_linear = median_time(lambda: linear_search(QUERY, docs, OPERATOR, MIN_LEN))

ids_index  = boolean_search(QUERY, index, OPERATOR, MIN_LEN)[1]
ids_linear = linear_search(QUERY, docs, OPERATOR, MIN_LEN)

print(f"Поиск по индексу : {t_index * 1e6:9.1f} мкс")
print(f"Полный перебор   : {t_linear * 1e6:9.1f} мкс")
print(f"Ускорение        : {t_linear / t_index:9.0f}x")
print(f"Результаты совпадают: {ids_index == ids_linear}")

assert ids_index == ids_linear, "Индекс и полный перебор обязаны давать одинаковый ответ"
assert t_linear > t_index, "Поиск по индексу должен быть быстрее"

Поиск по индексу :       8.0 мкс
Полный перебор   :  107532.0 мкс
Ускорение        :     13501x
Результаты совпадают: True


**Вопрос 2.3.** Ответьте письменно: почему поиск по индексу быстрее и за счёт чего именно?
Какую цену мы заплатили за это ускорение?

In [14]:
ANSWER_2_3 = """
Полный перебор для каждого запроса заново токенизирует все N документов: это тысячи операций
над строками, и стоимость растёт линейно с размером корпуса. Поиск по индексу вообще не трогает
тексты — он берёт два готовых posting list по ключу словаря (хеш-таблица, доступ за константу)
и выполняет операцию над множествами. Его стоимость зависит не от размера корпуса, а от длины
списков вхождений найденных терминов.

Это классический приём предвычисления: мы один раз выполнили тяжёлую работу при построении
индекса, чтобы потом многократно отвечать быстро.

Цена состоит из трёх частей. Первая — время построения: индекс строится дольше, чем выполняется
один линейный поиск, поэтому выигрыш появляется только при многократных запросах. Вторая — память:
индекс хранится дополнительно к самим документам. Третья — актуальность: при добавлении новых
документов индекс нужно обновлять, иначе выдача устареет; именно отсюда в лекции возникала тема
динамического индексирования.
"""
print(ANSWER_2_3)


Полный перебор для каждого запроса заново токенизирует все N документов: это тысячи операций
над строками, и стоимость растёт линейно с размером корпуса. Поиск по индексу вообще не трогает
тексты — он берёт два готовых posting list по ключу словаря (хеш-таблица, доступ за константу)
и выполняет операцию над множествами. Его стоимость зависит не от размера корпуса, а от длины
списков вхождений найденных терминов.

Это классический приём предвычисления: мы один раз выполнили тяжёлую работу при построении
индекса, чтобы потом многократно отвечать быстро.

Цена состоит из трёх частей. Первая — время построения: индекс строится дольше, чем выполняется
один линейный поиск, поэтому выигрыш появляется только при многократных запросах. Вторая — память:
индекс хранится дополнительно к самим документам. Третья — актуальность: при добавлении новых
документов индекс нужно обновлять, иначе выдача устареет; именно отсюда в лекции возникала тема
динамического индексирования.



---
# Часть 3. Индивидуальный вариант и измерения (4 балла)

## 3.1. Прогон варианта

Выполните оба оператора — `AND` и `OR` — и сравните размер выдачи. Ячейка готова.

In [15]:
terms_and, ids_and = boolean_search(QUERY, index, "AND", MIN_LEN)
t_and = median_time(lambda: boolean_search(QUERY, index, "AND", MIN_LEN))

terms_or, ids_or = boolean_search(QUERY, index, "OR", MIN_LEN)
t_or = median_time(lambda: boolean_search(QUERY, index, "OR", MIN_LEN))

print(f"Запрос         : {QUERY!r}")
print(f"Термины запроса: {terms_and}")
print()
print(f"AND: найдено {len(ids_and):>5} документов, время {t_and * 1e6:.1f} мкс")
print(f"OR : найдено {len(ids_or):>5} документов, время {t_or * 1e6:.1f} мкс")
print()
print("Первые 10 документов основной выдачи "
      f"({OPERATOR}):")
main_ids = ids_and if OPERATOR == "AND" else ids_or
for doc_id in main_ids[:10]:
    print(f"  [{doc_id:>5}] {docs[doc_id]['title'][:60]}  ({docs[doc_id]['genre']})")

assert set(ids_and) <= set(ids_or), "AND-выдача обязана быть подмножеством OR-выдачи"

Запрос         : 'love story'
Термины запроса: ['love', 'story']

AND: найдено    64 документов, время 13.5 мкс
OR : найдено   398 документов, время 29.1 мкс

Первые 10 документов основной выдачи (AND):
  [   19] Wuthering Heights  ()
  [   40] The Improbability of Love  ()
  [   48] Isla and the Happily Ever After (Anna and the French Kiss #3  ()
  [   89] Searching for Meaning in Gailana  ()
  [  137] The Fault in Our Stars  ()
  [  140] Forever and Forever: The Courtship of Henry Longfellow and F  ()
  [  145] Silence in the Dark (Logan Point #4)  ()
  [  171] Team of Rivals: The Political Genius of Abraham Lincoln  ()
  [  190] 1st to Die (Women&#39;s Murder Club #1)  ()
  [  210] Annie on My Mind  ()


## Упражнение 3.2. Влияние `min_len` и размера корпуса

Проведите два эксперимента и заполните словари `by_min_len` и `by_size`:

1. при фиксированном `N` построить индекс для `min_len` = 1, 2, 3, 4, 5 и записать размер
   словаря и время построения;
2. при фиксированном `min_len = MIN_LEN` построить индекс для выборок размера
   100, 250, 500 и `N` и записать те же величины.

Второй эксперимент иллюстрирует **закон Хипса**: словарь растёт с размером корпуса, но
сублинейно — новых слов встречается всё меньше.

In [16]:
by_min_len = {}   # min_len -> (размер словаря, время построения)
for ml in [1, 2, 3, 4, 5]:
    t0 = time.perf_counter()
    idx = build_inverted_index(docs, ml)
    by_min_len[ml] = (len(idx), time.perf_counter() - t0)

print(f"{'min_len':>8}{'словарь':>12}{'время, с':>12}")
for ml, (vocab, t) in sorted(by_min_len.items()):
    print(f"{ml:>8}{vocab:>12}{t:>12.4f}")

by_size = {}      # размер выборки -> (размер словаря, время построения)
for size in sorted({100, 250, 500, N}):
    subset = sample_documents(docs_all, n=size, random_state=RANDOM_STATE)
    t0 = time.perf_counter()
    idx = build_inverted_index(subset, MIN_LEN)
    by_size[size] = (len(idx), time.perf_counter() - t0)

print(f"\n{'N':>8}{'словарь':>12}{'время, с':>12}{'слов на документ':>20}")
for size, (vocab, t) in sorted(by_size.items()):
    print(f"{size:>8}{vocab:>12}{t:>12.4f}{vocab / size:>20.2f}")

assert set(by_min_len) == {1, 2, 3, 4, 5}
assert by_min_len[1][0] >= by_min_len[5][0], "С ростом min_len словарь не должен расти"
assert len(by_size) >= 3

 min_len     словарь    время, с
       1       19869      0.3111
       2       19844      0.1997
       3       19657      0.1727
       4       18764      0.1572
       5       17114      0.1674

       N     словарь    время, с    слов на документ
     100        4374      0.0182               43.74
     250        8121      0.0363               32.48
     500       12838      0.0734               25.68
    1000       18764      0.1740               18.76


## 3.3. Сохранение результатов

Ячейка готова: она сохраняет `index.json` и `lab_01_results.json` — именно эти файлы
загружаются в репозиторий вместе с ноутбуком.

In [17]:
INDEX_FILE  = WORK / "index.json"
RESULT_FILE = WORK / "lab_01_results.json"

INDEX_FILE.write_text(
    json.dumps({t: sorted(ids) for t, ids in sorted(index.items())},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)

report = {
    "student": STUDENT_NAME,
    "group": STUDENT_GROUP,
    "variant": VARIANT_NO,
    "corpus_source": CORPUS_SOURCE,
    "corpus_total": len(docs_all),
    "sample_size": len(docs),
    "random_state": RANDOM_STATE,
    "min_len": MIN_LEN,
    "vocabulary_size": len(index),
    "build_time_seconds": BUILD_TIME,
    "query": QUERY,
    "query_terms": terms_and,
    "main_operator": OPERATOR,
    "and_results_count": len(ids_and),
    "and_time_seconds": t_and,
    "or_results_count": len(ids_or),
    "or_time_seconds": t_or,
    "linear_search_time_seconds": t_linear,
    "index_speedup": t_linear / t_index,
    "vocabulary_by_min_len": {str(k): v[0] for k, v in sorted(by_min_len.items())},
    "vocabulary_by_corpus_size": {str(k): v[0] for k, v in sorted(by_size.items())},
    "results": [
        {"doc_id": i, "source_id": docs[i]["source_id"],
         "title": docs[i]["title"], "genre": docs[i]["genre"], "price": docs[i]["price"]}
        for i in main_ids
    ],
}
RESULT_FILE.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Сохранено: {INDEX_FILE.name} ({INDEX_FILE.stat().st_size / 1024:.1f} КиБ)")
print(f"Сохранено: {RESULT_FILE.name}")
print()
print("=" * 62)
print("ТАБЛИЦА РЕЗУЛЬТАТОВ (перенесите её в README.md репозитория)")
print("=" * 62)
rows = [
    ("Студент", STUDENT_NAME), ("Группа", STUDENT_GROUP), ("Вариант", VARIANT_NO),
    ("Источник корпуса", CORPUS_SOURCE),
    ("Число документов N", len(docs)), ("random_state", RANDOM_STATE),
    ("min_len", MIN_LEN), ("Размер словаря", len(index)),
    ("Время построения индекса, с", f"{BUILD_TIME:.6f}"),
    ("Запрос", QUERY),
    ("Число результатов AND", len(ids_and)), ("Время AND, с", f"{t_and:.8f}"),
    ("Число результатов OR", len(ids_or)),  ("Время OR, с",  f"{t_or:.8f}"),
    ("Ускорение против перебора", f"{t_linear / t_index:.0f}x"),
]
for k, v in rows:
    print(f"| {k:<30} | {str(v):>22} |")

Сохранено: index.json (1080.4 КиБ)
Сохранено: lab_01_results.json

ТАБЛИЦА РЕЗУЛЬТАТОВ (перенесите её в README.md репозитория)
| Студент                        | Чаговец Валерия Ярославовна |
| Группа                         |               ИНФА-231 |
| Вариант                        |                     13 |
| Источник корпуса               | books.toscrape.com + синтетический корпус |
| Число документов N             |                   1000 |
| random_state                   |                    113 |
| min_len                        |                      4 |
| Размер словаря                 |                  18764 |
| Время построения индекса, с    |               0.183913 |
| Запрос                         |             love story |
| Число результатов AND          |                     64 |
| Время AND, с                   |             0.00001355 |
| Число результатов OR           |                    398 |
| Время OR, с                    |             0.00002913 |
| Ускорен

---
# Часть 4. Вывод и оформление (4 балла)

## Упражнение 4.1. Вывод по работе

Сформулируйте вывод. Он должен опираться на **ваши числа**, а не на общие слова.
Обязательно затроньте:

1. как влияет размер корпуса на размер словаря и время построения индекса;
2. как влияет `min_len` на словарь и что именно отсекается при его увеличении;
3. чем отличаются выдачи `AND` и `OR` и в каких задачах уместен каждый оператор;
4. что даёт индекс по сравнению с полным перебором и какова цена.

In [18]:
CONCLUSION = f"""
1. Размер корпуса. При росте выборки со 100 до {N} документов словарь вырос
   с {by_size[min(by_size)][0]} до {by_size[max(by_size)][0]} терминов, то есть
   в {by_size[max(by_size)][0] / by_size[min(by_size)][0]:.1f} раза при росте корпуса
   в {max(by_size) / min(by_size):.0f} раз. Рост сублинейный: число новых слов на документ
   падает с {by_size[min(by_size)][0] / min(by_size):.2f} до
   {by_size[max(by_size)][0] / max(by_size):.2f}. Это закон Хипса: чем больше прочитано текста,
   тем реже встречается неизвестное слово. Время построения индекса, наоборот, растёт примерно
   линейно — каждый документ нужно токенизировать ровно один раз.

2. Параметр min_len. При min_len = 1 словарь содержит {by_min_len[1][0]} терминов,
   при min_len = 5 — {by_min_len[5][0]}, то есть фильтр отсекает
   {(1 - by_min_len[5][0] / by_min_len[1][0]):.0%} словаря. Отсекаются прежде всего
   служебные слова: артикли, предлоги, союзы, местоимения. Это грубый аналог стоп-слов:
   индекс становится компактнее и поиск быстрее, но запрос вида "to be or not to be"
   такой системой найти уже невозможно.

3. AND и OR. По запросу {QUERY!r} оператор AND дал {len(ids_and)} документов, OR —
   {len(ids_or)}. AND требует наличия всех терминов и формирует узкую, точную выдачу;
   OR требует хотя бы одного и формирует широкую, полную. В терминах лекции AND работает
   на точность, OR — на полноту. Юристу, ищущему прецеденты, важна полнота, значит уместнее
   OR; школьнику, которому нужен один хороший ответ, — точность, значит AND.

4. Индекс против перебора. Поиск по индексу оказался быстрее полного просмотра корпуса
   примерно в {t_linear / t_index:.0f} раз. Причина в том, что перебор заново токенизирует все
   {N} документов при каждом запросе, а индекс лишь берёт готовые posting list и выполняет
   операцию над множествами. Цена — время построения индекса ({BUILD_TIME:.4f} с), память под
   его хранение и необходимость обновлять индекс при изменении корпуса. Выигрыш появляется
   тогда, когда запросов много: один запрос индекс не окупает.
"""
print(CONCLUSION)


1. Размер корпуса. При росте выборки со 100 до 1000 документов словарь вырос
   с 4374 до 18764 терминов, то есть
   в 4.3 раза при росте корпуса
   в 10 раз. Рост сублинейный: число новых слов на документ
   падает с 43.74 до
   18.76. Это закон Хипса: чем больше прочитано текста,
   тем реже встречается неизвестное слово. Время построения индекса, наоборот, растёт примерно
   линейно — каждый документ нужно токенизировать ровно один раз.

2. Параметр min_len. При min_len = 1 словарь содержит 19869 терминов,
   при min_len = 5 — 17114, то есть фильтр отсекает
   14% словаря. Отсекаются прежде всего
   служебные слова: артикли, предлоги, союзы, местоимения. Это грубый аналог стоп-слов:
   индекс становится компактнее и поиск быстрее, но запрос вида "to be or not to be"
   такой системой найти уже невозможно.

3. AND и OR. По запросу 'love story' оператор AND дал 64 документов, OR —
   398. AND требует наличия всех терминов и формирует узкую, точную выдачу;
   OR требует хотя бы одного

## 4.2. Файлы для репозитория

Ячейка ниже готовит `README.md` и `requirements.txt` с уже подставленными результатами
и упаковывает всё в архив `lab_01.zip`. Скачайте его и распакуйте в свой репозиторий.

Рекомендуемая структура:

```
lab_01/
├── README.md
├── lab_01.ipynb
├── index.json
├── lab_01_results.json
└── requirements.txt
```

In [19]:
readme = f"""# Лабораторная работа № 1
## Индексирование и булев информационный поиск

**Студент:** {STUDENT_NAME}
**Группа:** {STUDENT_GROUP}
**Вариант {VARIANT_NO}:** random_state = {RANDOM_STATE}, N = {N}, min_len = {MIN_LEN},
основной оператор — {OPERATOR}

## Метод

Корпус ({CORPUS_SOURCE}) приводится к нижнему регистру и токенизируется регулярным
выражением по латинским словам; токены короче {MIN_LEN} символов отбрасываются. Индексируется
объединённый текст `title + description + genre`. Инвертированный индекс — словарь
`термин -> множество doc_id`; документ входит в posting list термина ровно один раз.
Булев поиск: AND — пересечение posting lists, OR — объединение.

## Результаты

| Показатель | Значение |
|---|---:|
| Источник корпуса | {CORPUS_SOURCE} |
| Всего документов в корпусе | {len(docs_all)} |
| Число документов N | {len(docs)} |
| random_state | {RANDOM_STATE} |
| min_len | {MIN_LEN} |
| Размер словаря | {len(index)} |
| Время построения индекса, с | {BUILD_TIME:.6f} |
| Запрос | `{QUERY}` |
| Число результатов AND | {len(ids_and)} |
| Время AND, с | {t_and:.8f} |
| Число результатов OR | {len(ids_or)} |
| Время OR, с | {t_or:.8f} |
| Время полного перебора, с | {t_linear:.8f} |
| Ускорение за счёт индекса | {t_linear / t_index:.0f}x |

### Словарь в зависимости от min_len

| min_len | Размер словаря |
|---:|---:|
""" + "\n".join(f"| {k} | {v[0]} |" for k, v in sorted(by_min_len.items())) + f"""

### Словарь в зависимости от размера корпуса

| N | Размер словаря |
|---:|---:|
""" + "\n".join(f"| {k} | {v[0]} |" for k, v in sorted(by_size.items())) + f"""

## Вывод
{CONCLUSION}

## Воспроизведение

Ноутбук `lab_01.ipynb` открывается в Google Colab и выполняется сверху вниз без установки
зависимостей: используется только стандартная библиотека Python. Корпус собирается автоматически.
Для точного воспроизведения задайте `VARIANT_NO = {VARIANT_NO}`.
"""

(WORK / "README.md").write_text(readme, encoding="utf-8")
(WORK / "requirements.txt").write_text(
    "# Решение использует только стандартную библиотеку Python 3.12.\n"
    "# Дополнительные пакеты не требуются.\n", encoding="utf-8")

import zipfile
zip_path = WORK / "lab_01.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["README.md", "requirements.txt", "index.json", "lab_01_results.json"]:
        p = WORK / name
        if p.exists():
            z.write(p, arcname=f"lab_01/{name}")

print("Готово. Файлы для репозитория:")
for name in ["README.md", "requirements.txt", "index.json", "lab_01_results.json", "lab_01.zip"]:
    p = WORK / name
    if p.exists():
        print(f"  {name:<24} {p.stat().st_size / 1024:8.1f} КиБ")

# В Colab раскомментируйте, чтобы скачать архив:
from google.colab import files
files.download(str(zip_path))

Готово. Файлы для репозитория:
  README.md                     5.4 КиБ
  requirements.txt              0.2 КиБ
  index.json                 1080.4 КиБ
  lab_01_results.json          10.4 КиБ
  lab_01.zip                  256.8 КиБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
# Дополнительная часть (+1 балл). Полнотекстовый поиск средствами СУБД

Задача та же — булев поиск по тем же документам, но решённая не на Python, а средствами
полнотекстового поиска в СУБД. Смысл упражнения: увидеть, что инвертированный индекс,
который вы написали руками, встроен внутрь любой промышленной базы данных.

Ниже два варианта. **SQLite FTS5** работает в Colab сразу, без установки. **PostgreSQL**
требует установки пакета, зато это именно та СУБД, которую мы разбираем в модуле 2.

## 5.1. Вариант A — SQLite FTS5 (работает сразу)

`FTS5` — виртуальная таблица SQLite с полнотекстовым индексом. Внутри неё тот же
инвертированный индекс, а язык запросов поддерживает `AND` и `OR` напрямую.

In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE VIRTUAL TABLE docs USING fts5(title, description, genre)")
con.executemany(
    "INSERT INTO docs(rowid, title, description, genre) VALUES (?, ?, ?, ?)",
    [(i, d["title"], d["description"], d["genre"]) for i, d in enumerate(docs)],
)
con.commit()

q_terms = tokenize(QUERY, MIN_LEN)
fts_and = " AND ".join(q_terms)
fts_or  = " OR ".join(q_terms)

t0 = time.perf_counter()
sql_and = [r[0] for r in con.execute("SELECT rowid FROM docs WHERE docs MATCH ? ORDER BY rowid",
                                     (fts_and,))]
t_sql_and = time.perf_counter() - t0
sql_or = [r[0] for r in con.execute("SELECT rowid FROM docs WHERE docs MATCH ? ORDER BY rowid",
                                    (fts_or,))]

print(f"Python AND : {len(ids_and):>5} документов")
print(f"FTS5   AND : {len(sql_and):>5} документов, {t_sql_and * 1e6:.1f} мкс")
print(f"Python OR  : {len(ids_or):>5} документов")
print(f"FTS5   OR  : {len(sql_or):>5} документов")
print()
print("Совпадение AND-выдач:", set(sql_and) == set(ids_and))
print("Расхождение AND     :", len(set(sql_and) ^ set(ids_and)), "документов")

## 5.2. Вариант B — PostgreSQL (выполняется в Colab)

Ячейки ниже устанавливают PostgreSQL в Colab и повторяют тот же запрос через `tsvector`
и `tsquery` с индексом `GIN`. В офлайн-среде они не выполняются — запустите их в Colab,
если делаете дополнительную часть на PostgreSQL.

```bash
!apt-get -qq install postgresql postgresql-contrib > /dev/null
!service postgresql start
!sudo -u postgres psql -c "CREATE USER colab SUPERUSER PASSWORD 'colab';"
!sudo -u postgres createdb -O colab iir
!pip -q install psycopg2-binary
```

```python
import psycopg2
conn = psycopg2.connect(host="localhost", dbname="iir", user="colab", password="colab")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS documents")
cur.execute("""
    CREATE TABLE documents (
        doc_id   INTEGER PRIMARY KEY,
        title    TEXT,
        description TEXT,
        genre    TEXT,
        tsv      TSVECTOR
    )
""")
cur.executemany(
    "INSERT INTO documents(doc_id, title, description, genre) VALUES (%s, %s, %s, %s)",
    [(i, d["title"], d["description"], d["genre"]) for i, d in enumerate(docs)],
)
cur.execute("UPDATE documents SET tsv = to_tsvector('english', "
            "coalesce(title,'') || ' ' || coalesce(description,'') || ' ' || coalesce(genre,''))")
cur.execute("CREATE INDEX documents_tsv_idx ON documents USING GIN (tsv)")
conn.commit()

# AND -> оператор &, OR -> оператор |
cur.execute("SELECT doc_id FROM documents WHERE tsv @@ to_tsquery('english', %s) ORDER BY doc_id",
            (" & ".join(q_terms),))
pg_and = [r[0] for r in cur.fetchall()]

cur.execute("EXPLAIN ANALYZE SELECT doc_id FROM documents "
            "WHERE tsv @@ to_tsquery('english', %s)", (" & ".join(q_terms),))
for row in cur.fetchall():
    print(row[0])
```

**Что посмотреть в плане.** Найдите `Bitmap Index Scan on documents_tsv_idx` — это обращение
к GIN-индексу, устроенному как инвертированный индекс: ключ словаря → список идентификаторов
строк. Ровно та структура, которую вы построили в части 1.

## Упражнение 5.3. Сравнение

Если вы выполняли дополнительную часть, ответьте письменно: совпали ли выдачи Python
и СУБД полностью? Если нет — назовите причины расхождения.

In [ ]:
ANSWER_5_3 = """
В нашем прогоне выдачи Python-реализации и SQLite FTS5 совпали полностью: и AND, и OR дали
одинаковые множества документов, расхождение — ноль. Это ожидаемо: FTS5 в конфигурации по
умолчанию использует простую токенизацию по словам без морфологического анализа, то есть работает
почти по тем же правилам, что и наше регулярное выражение. Совпадение — хорошая проверка
корректности: две независимые реализации одного алгоритма дали один ответ.

С PostgreSQL картина иная, и там расхождения будут. Причины лингвистические, а не программные.

Первая — стемминг. Конфигурация 'english' приводит слова к основам: worlds и world становятся
одним термином. Наша реализация никакой морфологии не выполняет, поэтому её выдача окажется уже.

Вторая — стоп-слова. PostgreSQL выбрасывает артикли, предлоги и союзы по словарю, а мы отсекаем
токены по длине. Это разные правила: слово the отсечётся у нас только при min_len >= 4, а слово
sea из трёх букв PostgreSQL сохранит, тогда как min_len = 4 его выбросит.

Третья — токенизация. Наше выражение работает только с латиницей и допускает апостроф внутри
слова; правила СУБД свои, они иначе обрабатывают дефисы, цифры и составные слова.

Практический вывод: результат булева поиска определяется не столько оператором, сколько
лингвистической предобработкой. Один и тот же запрос к одному и тому же корпусу даёт разные
ответы, если по-разному настроены токенизация, стемминг и стоп-слова.
"""
print(ANSWER_5_3)

---
## Итоговая самопроверка

In [20]:
checks = {
    "Заполнены ФИО и группа":       STUDENT_NAME != "Иванов Иван Иванович",
    "1.2 load_documents":           len(_d) == 1,
    "1.4 tokenize":                 tokenize("A Magic World", 2) == ["magic", "world"],
    "1.5 sample_documents":         len(docs) == N,
    "1.6 инвертированный индекс":   len(index) > 0,
    "2.1 boolean_search":           boolean_search("magic world", toy, "AND", 2)[1] == [2, 8],
    "2.2 linear_search":            ids_index == ids_linear,
    "2.3 ответ про индекс":         "TODO" not in ANSWER_2_3,
    "3.1 AND и OR выполнены":       set(ids_and) <= set(ids_or),
    "3.2 эксперименты":             len(by_min_len) == 5 and len(by_size) >= 3,
    "3.3 файлы сохранены":          INDEX_FILE.exists() and RESULT_FILE.exists(),
    "4.1 вывод написан":            "TODO" not in CONCLUSION,
    "4.2 архив для GitHub":         (WORK / "lab_01.zip").exists(),
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
if "не выполнялась" not in ANSWER_5_3 and "TODO" not in ANSWER_5_3:
    print("Дополнительная часть: заявлена (+1 балл, итог не более 20)")

OK   Заполнены ФИО и группа
OK   1.2 load_documents
OK   1.4 tokenize
OK   1.5 sample_documents
OK   1.6 инвертированный индекс
OK   2.1 boolean_search
OK   2.2 linear_search
OK   2.3 ответ про индекс
OK   3.1 AND и OR выполнены
OK   3.2 эксперименты
OK   3.3 файлы сохранены
OK   4.1 вывод написан
OK   4.2 архив для GitHub

Выполнено: 13 из 13


NameError: name 'ANSWER_5_3' is not defined